# Módulo 3 · Fundamentos de Deep Learning

## Objetivos
1. Implementar una **red neuronal desde cero en NumPy** (forward, pérdida, *backpropagation*, actualización) y verificarla con *gradient check*.
2. Pasar a **PyTorch 2.x**: `autograd`, `nn.Module`, `DataLoader`, bucle de entrenamiento, `torch.compile`, precisión mixta y detección automática del acelerador.
3. Dominar la **caja de herramientas de optimización y regularización**: inicialización, normalización (BatchNorm/LayerNorm), dropout, *weight decay*, AdamW, *schedulers*, *early stopping*.
4. Entender por qué las **CNN** funcionan (localidad, pesos compartidos, invariancia) y entrenar una pequeña.
5. Entender las **RNN/GRU/LSTM**, el problema del gradiente que se desvanece y por qué los Transformers las reemplazaron (Módulo 5).
6. Conocer el ecosistema 2026: PyTorch como estándar de investigación e industria, **Keras 3** multi-backend (JAX/TF/PyTorch), **JAX** para investigación a gran escala.

## Mapa de lectura (capítulos verificados)
| Tema | Libro | Capítulo verificado |
|---|---|---|
| Fundamentos matemáticos y una red desde cero | Weidman, *Deep Learning from Scratch* | **Cap. 1** Foundations · **Cap. 2** Fundamentals · **Cap. 3** Deep Learning from Scratch · **Cap. 4** Extensions (optimizadores, dropout, inicialización) |
| Convoluciones y RNN desde cero | Weidman | **Cap. 5** Convolutions · **Cap. 6** RNNs · **Cap. 7** PyTorch |
| Regresión lineal/softmax como redes, MLP, guía del constructor | Zhang et al., *Dive into Deep Learning* | **Cap. 3** Linear Regression · **Cap. 4** Linear Classification · **Cap. 5** Multilayer Perceptrons · **Cap. 6** Builders' Guide |
| CNN clásicas y modernas | D2L | **Cap. 7** Convolutional Neural Networks · **Cap. 8** Modern CNNs (ResNet, BatchNorm, DenseNet) |
| RNN, GRU, LSTM | D2L | **Cap. 9** Recurrent Neural Networks · **Cap. 10** Modern RNNs |
| Algoritmos de optimización | D2L | **Cap. 12** Optimization Algorithms |
| Keras, entrenamiento de redes profundas, CNN, secuencias | Géron, *Hands-On ML* (3.ª ed.) | **Cap. 10** Introduction to ANNs with Keras · **Cap. 11** Training Deep Neural Networks · **Cap. 14** Deep Computer Vision Using CNNs · **Cap. 15** Processing Sequences Using RNNs and CNNs |

> **Nota de actualización (2026).** La malla original citaba "Zhang et al. Cap. 2" para redes neuronales; en D2L el Cap. 2 son *Preliminaries* (NumPy-like, autograd). Las redes están en Cap. 3-6. Se hace de **PyTorch el framework principal** (es el usado por >90 % de los artículos y por Hugging Face); TensorFlow/Keras se mantiene vía **Keras 3**, que hoy corre sobre JAX, TensorFlow o PyTorch con el mismo código.

> **Ruta de aprendizaje Intensiva en ML Aplicado y DL Moderno** · Notebook del **Módulo 3 · Fundamentos de Deep Learning**
> Licencia CC BY-NC-SA 4.0 · © 2025-2026 Alexander Saravia
>
> Cómo usar este notebook: léelo de arriba hacia abajo. Cada sección tiene (1) la **idea** explicada con palabras y fórmulas,
> (2) **código ejecutable** que la materializa desde cero o con la librería estándar de 2026, (3) **lectura guiada** con
> el capítulo *verificado* de la bibliografía y (4) **ejercicios**. Las celdas marcadas `# OPCIONAL` requieren dependencias
> pesadas (GPU, Hugging Face, Spark) y están protegidas con `try/except` para que el resto del notebook siga corriendo.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time
rng = np.random.default_rng(0)

## 1. Una red neuronal desde cero (Weidman Cap. 2-3; D2L Cap. 5)

Un **perceptrón multicapa (MLP)** de dos capas para clasificación binaria:

$$\mathbf{h} = \phi(W_1\mathbf{x}+\mathbf{b}_1),\qquad \hat p = \sigma(\mathbf{w}_2^\top\mathbf{h}+b_2),$$

con $\phi=\mathrm{ReLU}$ y $\sigma$ la sigmoide. La pérdida es la **entropía cruzada binaria**
$$\mathcal{L} = -\frac1n\sum_i\big[y_i\log\hat p_i + (1-y_i)\log(1-\hat p_i)\big].$$

**Backpropagation** = regla de la cadena en orden inverso. La clave que simplifica todo: para sigmoide + entropía cruzada, $\partial\mathcal{L}/\partial z_2 = (\hat p - y)/n$. A partir de ahí, cada capa recibe el gradiente de su salida y devuelve (a) el gradiente de sus parámetros y (b) el gradiente de su entrada para la capa anterior.

In [ ]:
def relu(z): return np.maximum(z, 0)
def sigmoid(z): return 1 / (1 + np.exp(-z))

class MLP:
    def __init__(self, d_in, d_h, seed=0):
        r = np.random.default_rng(seed)
        # Inicialización de He (ReLU): Var(W) = 2/fan_in evita que la señal se desvanezca o explote
        self.W1 = r.normal(0, np.sqrt(2 / d_in), (d_in, d_h)); self.b1 = np.zeros(d_h)
        self.W2 = r.normal(0, np.sqrt(1 / d_h), (d_h, 1));     self.b2 = np.zeros(1)

    def forward(self, X):
        self.X = X
        self.Z1 = X @ self.W1 + self.b1;  self.H = relu(self.Z1)
        self.Z2 = self.H @ self.W2 + self.b2; self.P = sigmoid(self.Z2)
        return self.P

    def loss(self, P, y):
        eps = 1e-12
        return -np.mean(y * np.log(P + eps) + (1 - y) * np.log(1 - P + eps))

    def backward(self, y):
        n = len(y)
        dZ2 = (self.P - y[:, None]) / n                 # ∂L/∂Z2  (sigmoide + BCE se simplifica)
        self.dW2 = self.H.T @ dZ2; self.db2 = dZ2.sum(0)
        dH = dZ2 @ self.W2.T                            # ∂L/∂H
        dZ1 = dH * (self.Z1 > 0)                        # ∂L/∂Z1 (derivada de ReLU)
        self.dW1 = self.X.T @ dZ1; self.db1 = dZ1.sum(0)

    def step(self, lr):
        for p, g in [("W1", "dW1"), ("b1", "db1"), ("W2", "dW2"), ("b2", "db2")]:
            setattr(self, p, getattr(self, p) - lr * getattr(self, g))

# Gradient check: la prueba definitiva de que backprop está bien
X = rng.normal(size=(8, 3)); y = rng.integers(0, 2, 8).astype(float)
net = MLP(3, 5); net.forward(X); net.backward(y)
h = 1e-6; i, j = 1, 2
W_plus = net.W1.copy(); W_plus[i, j] += h; W_minus = net.W1.copy(); W_minus[i, j] -= h
net_p = MLP(3, 5); net_p.W1 = W_plus; net_m = MLP(3, 5); net_m.W1 = W_minus
num = (net_p.loss(net_p.forward(X), y) - net_m.loss(net_m.forward(X), y)) / (2 * h)
print(f"∂L/∂W1[{i},{j}]  analítico={net.dW1[i, j]:.8f}  numérico={num:.8f}")

In [ ]:
from sklearn.datasets import make_moons
Xm, ym = make_moons(n_samples=1000, noise=0.2, random_state=0)
Xtr, Xte, ytr, yte = Xm[:700], Xm[700:], ym[:700].astype(float), ym[700:].astype(float)

net = MLP(2, 32); hist = []
for epoch in range(3000):
    P = net.forward(Xtr); hist.append(net.loss(P, ytr)); net.backward(ytr); net.step(lr=0.5)
acc = ((net.forward(Xte) > 0.5).ravel() == yte).mean()
print(f"exactitud test MLP-NumPy: {acc:.3f}")

xx, yy = np.meshgrid(np.linspace(-1.8, 2.8, 200), np.linspace(-1.3, 1.8, 200))
Z = net.forward(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(hist); axes[0].set_yscale("log"); axes[0].set_title("pérdida de entrenamiento")
axes[1].contourf(xx, yy, Z, levels=20, cmap="RdBu", alpha=0.7); axes[1].scatter(*Xte.T, c=yte, cmap="RdBu", edgecolor="k", s=15)
axes[1].set_title("frontera de decisión no lineal aprendida"); plt.show()

## 2. PyTorch 2.x: autograd y el bucle de entrenamiento canónico

PyTorch construye el grafo de cómputo **dinámicamente** mientras ejecutas operaciones sobre tensores con `requires_grad=True`; `loss.backward()` recorre ese grafo y acumula gradientes en `.grad`. Encima de esto:

* `nn.Module` — contenedor de parámetros y submódulos.
* `torch.optim` — SGD, **AdamW** (Adam con *weight decay* desacoplado, el estándar), etc.
* `DataLoader` — lotes, barajado, carga paralela.
* `torch.compile(model)` (PyTorch ≥ 2.0) — compila el grafo (TorchDynamo + Inductor) y suele dar 1.3-2× de aceleración sin cambiar el código.
* `torch.autocast` + `GradScaler` — precisión mixta (bf16/fp16) para entrenar más rápido y con menos memoria en GPU.

El bucle que sigue es el que verás en el 95 % de los proyectos; apréndelo de memoria.

In [ ]:
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
    TORCH = True
    # Detección moderna del acelerador (PyTorch >= 2.5: torch.accelerator). Fallback para versiones previas.
    if hasattr(torch, "accelerator") and torch.accelerator.is_available():
        device = torch.accelerator.current_accelerator()
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    elif getattr(torch.backends, "mps", None) and torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")
    torch.manual_seed(0)
    print("PyTorch", torch.__version__, "| dispositivo:", device)
except ImportError:
    TORCH = False
    print("PyTorch no está instalado: `uv pip install torch`. Las celdas siguientes se omiten.")

In [ ]:
if TORCH:
    # Autograd en una línea: gradiente de f(w) = sum((Xw - y)^2) respecto a w
    Xt = torch.randn(50, 3); w = torch.randn(3, requires_grad=True); yt = torch.randn(50)
    loss = ((Xt @ w - yt) ** 2).sum()
    loss.backward()
    grad_manual = 2 * Xt.T @ (Xt @ w.detach() - yt)
    print("autograd vs manual, error máx:", (w.grad - grad_manual).abs().max().item())

In [ ]:
if TORCH:
    def entrenar(model, train_dl, val_dl, epochs=20, lr=1e-3, weight_decay=1e-4, verbose=True):
        model = model.to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=lr, total_steps=epochs * len(train_dl))
        hist = {"train": [], "val": [], "val_acc": []}
        for ep in range(epochs):
            model.train(); tot = 0.0
            for xb, yb in train_dl:
                xb, yb = xb.to(device), yb.to(device)
                opt.zero_grad(set_to_none=True)
                loss = F.cross_entropy(model(xb), yb)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)   # estabilidad
                opt.step(); sched.step(); tot += loss.item() * len(xb)
            hist["train"].append(tot / len(train_dl.dataset))
            model.eval(); vl, correct = 0.0, 0
            with torch.no_grad():
                for xb, yb in val_dl:
                    xb, yb = xb.to(device), yb.to(device); out = model(xb)
                    vl += F.cross_entropy(out, yb, reduction="sum").item(); correct += (out.argmax(1) == yb).sum().item()
            hist["val"].append(vl / len(val_dl.dataset)); hist["val_acc"].append(correct / len(val_dl.dataset))
            if verbose and (ep % 5 == 0 or ep == epochs - 1):
                print(f"época {ep:3d}  train={hist['train'][-1]:.4f}  val={hist['val'][-1]:.4f}  acc={hist['val_acc'][-1]:.3f}")
        return hist

    # Datos: dígitos 8x8 de sklearn (1797 imágenes) -> rápido en CPU
    from sklearn.datasets import load_digits
    from sklearn.model_selection import train_test_split
    d = load_digits(); Xd = (d.data / 16.0).astype(np.float32); yd = d.target
    Xa, Xb_, ya, yb_ = train_test_split(Xd, yd, test_size=0.25, stratify=yd, random_state=0)
    mk = lambda X, y, sh: DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)), batch_size=64, shuffle=sh)
    train_dl, val_dl = mk(Xa, ya, True), mk(Xb_, yb_, False)

    mlp = nn.Sequential(nn.Linear(64, 128), nn.ReLU(), nn.Dropout(0.2), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))
    print("parámetros:", sum(p.numel() for p in mlp.parameters()))
    h_mlp = entrenar(mlp, train_dl, val_dl, epochs=20, lr=3e-3)

## 3. La caja de herramientas: por qué cada pieza existe (D2L Cap. 6, 12; Géron Cap. 11)

| Problema | Herramienta | Idea |
|---|---|---|
| Señal se desvanece/explota al inicio | **Inicialización** He/Xavier | mantener la varianza de activaciones y gradientes ≈ constante por capa |
| Distribución de activaciones cambia durante el entrenamiento | **BatchNorm** (CNN) / **LayerNorm** (Transformers) | normalizar y re-escalar con parámetros aprendibles |
| Sobreajuste | **Dropout**, *weight decay*, *data augmentation*, *early stopping* | ruido/penalización que fuerza representaciones robustas |
| Tasa de aprendizaje fija subóptima | **Schedulers**: OneCycle, cosine con *warmup* | grande al inicio para explorar, pequeño al final para converger |
| Gradientes ruidosos / escalas distintas por parámetro | **Adam/AdamW** | momento + normalización adaptativa por coordenada; AdamW desacopla el *weight decay* |
| Gradientes gigantes en RNN/Transformers | **Gradient clipping** | recortar la norma del gradiente |
| Redes muy profundas no entrenan | **Conexiones residuales** (ResNet, D2L 8.6) | $x_{l+1}=x_l+F(x_l)$: la identidad es fácil de aprender |

La siguiente celda muestra empíricamente el efecto de la inicialización sobre la varianza de las activaciones en una red profunda *sin* normalización.

In [ ]:
if TORCH:
    def varianzas_por_capa(init, n_capas=30, d=256):
        x = torch.randn(1000, d); vs = []
        for _ in range(n_capas):
            W = torch.empty(d, d); init(W)
            x = torch.relu(x @ W); vs.append(x.var().item())
        return vs
    inits = {"N(0, 0.01)": lambda W: nn.init.normal_(W, 0, 0.01),
             "N(0, 1/√d) (Xavier-ish)": lambda W: nn.init.normal_(W, 0, (1 / 256) ** 0.5),
             "He  N(0, √(2/d))": lambda W: nn.init.normal_(W, 0, (2 / 256) ** 0.5)}
    for k, f in inits.items(): plt.plot(varianzas_por_capa(f), label=k)
    plt.yscale("log"); plt.xlabel("capa"); plt.ylabel("varianza de activaciones"); plt.legend()
    plt.title("Sin una buena inicialización la señal muere o explota (ReLU, 30 capas)"); plt.show()

## 4. Redes convolucionales (D2L Cap. 7-8; Weidman Cap. 5; Géron Cap. 14)

Una capa convolucional aplica el **mismo** filtro pequeño $K$ (p. ej. 3×3) en todas las posiciones:
$$ (X * K)[i,j] = \sum_{a,b} X[i+a, j+b]\,K[a,b]. $$
Consecuencias: (1) **muchos menos parámetros** que una capa densa (una capa densa 64→128 sobre 8×8 tiene 8 k pesos; un filtro 3×3 tiene 9); (2) **equivarianza a traslación**: un borde se detecta igual en cualquier lugar; (3) apilando capas + *pooling* crece el **campo receptivo** y la jerarquía de características (bordes → texturas → partes → objetos).

Arquitecturas clave que debes reconocer: LeNet → AlexNet → VGG → **ResNet** (residuales + BatchNorm) → ConvNeXt (2022, CNN "modernizada" que iguala a los Vision Transformers).

In [ ]:
if TORCH:
    # Convolución a mano vs F.conv2d para entender la operación
    img = torch.tensor(Xa[0]).view(1, 1, 8, 8)
    sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
    out = F.conv2d(img, sobel_x, padding=1)
    manual = torch.zeros(8, 8); pad = F.pad(img, (1, 1, 1, 1))[0, 0]
    for i in range(8):
        for j in range(8):
            manual[i, j] = (pad[i:i+3, j:j+3] * sobel_x[0, 0]).sum()
    print("conv manual == F.conv2d:", torch.allclose(out[0, 0], manual, atol=1e-5))
    fig, axes = plt.subplots(1, 2, figsize=(6, 3)); axes[0].imshow(img[0, 0], cmap="gray"); axes[0].set_title("dígito")
    axes[1].imshow(out[0, 0], cmap="RdBu"); axes[1].set_title("bordes verticales (Sobel)"); [a.axis("off") for a in axes]; plt.show()

In [ ]:
if TORCH:
    class CNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv2d(1, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(),
                nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),   # 8x8 -> 4x4
                nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.AdaptiveAvgPool2d(1))
            self.head = nn.Sequential(nn.Flatten(), nn.Dropout(0.2), nn.Linear(64, 10))
        def forward(self, x):
            return self.head(self.features(x.view(-1, 1, 8, 8)))

    cnn = CNN(); print("parámetros CNN:", sum(p.numel() for p in cnn.parameters()))
    h_cnn = entrenar(cnn, train_dl, val_dl, epochs=20, lr=3e-3)

    plt.plot(h_mlp["val_acc"], label="MLP"); plt.plot(h_cnn["val_acc"], label="CNN")
    plt.xlabel("época"); plt.ylabel("exactitud validación"); plt.legend(); plt.title("MLP vs CNN en dígitos 8x8"); plt.show()

### `torch.compile`, precisión mixta y buenas prácticas de rendimiento
```python
model = torch.compile(model)                     # compila el grafo; la primera iteración es lenta (warm-up)
scaler = torch.amp.GradScaler()                  # solo necesario para fp16; bf16 no lo requiere
with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    loss = F.cross_entropy(model(xb), yb)
```
* Usa `bf16` en GPUs Ampere+ (A100/H100/RTX 30xx+); `channels_last` para CNN; `DataLoader(num_workers>0, pin_memory=True)`.
* Perfilado: `torch.profiler` o `torch.utils.benchmark`. Antes de optimizar, mide.

In [ ]:
if TORCH:
    # OPCIONAL: torch.compile en CPU puede tardar y requiere un compilador C++; lo probamos con protección
    try:
        cnn_c = torch.compile(CNN().to(device))
        xb, _ = next(iter(train_dl)); t0 = time.time(); cnn_c(xb.to(device)); print(f"compilación + 1ª llamada: {time.time()-t0:.1f}s")
        t0 = time.time(); [cnn_c(xb.to(device)) for _ in range(20)]; print(f"20 llamadas compiladas: {time.time()-t0:.3f}s")
    except Exception as e:
        print("torch.compile no disponible en este entorno:", type(e).__name__)

## 5. Redes recurrentes: RNN, GRU, LSTM (D2L Cap. 9-10; Weidman Cap. 6; Géron Cap. 15)

Una RNN procesa una secuencia manteniendo un **estado oculto**:
$$\mathbf{h}_t = \tanh(W_{xh}\mathbf{x}_t + W_{hh}\mathbf{h}_{t-1} + \mathbf{b}).$$
Al retropropagar a través de $T$ pasos aparece el producto $\prod_t W_{hh}^\top \mathrm{diag}(\tanh')$; si su norma es < 1 el gradiente **se desvanece** (no aprende dependencias largas), si es > 1 **explota**. Las **LSTM/GRU** añaden *compuertas* que permiten un camino casi lineal para el gradiente (la "cinta transportadora" de la celda). Aun así, procesan la secuencia **paso a paso** (no paralelizable) — la motivación directa de los Transformers.

Lo verificamos empíricamente: gradiente respecto a la entrada en $t=0$ en una RNN vs una GRU.

In [ ]:
if TORCH:
    T, d = 60, 16
    def grad_en_t0(cell):
        x = torch.randn(1, T, d, requires_grad=True)
        out, _ = cell(x)
        out[0, -1].sum().backward()                       # señal solo desde el último paso
        return x.grad[0].norm(dim=1).detach().numpy()     # norma del gradiente en cada t
    torch.manual_seed(1)
    g_rnn = grad_en_t0(nn.RNN(d, 32, batch_first=True)); g_gru = grad_en_t0(nn.GRU(d, 32, batch_first=True))
    plt.semilogy(g_rnn, label="RNN (tanh)"); plt.semilogy(g_gru, label="GRU"); plt.xlabel("paso t"); plt.ylabel("||∂y_T/∂x_t||")
    plt.legend(); plt.title("El gradiente hacia el pasado se desvanece mucho más rápido en la RNN simple"); plt.show()

In [ ]:
if TORCH:
    # Tarea de secuencia: predecir la suma de una señal binaria (cuenta de 1s) -> requiere memoria a lo largo de T
    def datos_seq(n, T=30):
        x = (torch.rand(n, T, 1) < 0.3).float(); y = x.sum(dim=(1, 2)).long().clamp(max=15)
        return x, y
    xs_tr, ys_tr = datos_seq(3000); xs_va, ys_va = datos_seq(600)
    seq_tr = DataLoader(TensorDataset(xs_tr, ys_tr), batch_size=64, shuffle=True); seq_va = DataLoader(TensorDataset(xs_va, ys_va), batch_size=256)

    class SeqClf(nn.Module):
        def __init__(self, cell):
            super().__init__(); self.rnn = cell; self.fc = nn.Linear(32, 16)
        def forward(self, x):
            out, _ = self.rnn(x); return self.fc(out[:, -1])
    for nombre, cell in [("RNN", nn.RNN(1, 32, batch_first=True)), ("GRU", nn.GRU(1, 32, batch_first=True))]:
        h = entrenar(SeqClf(cell), seq_tr, seq_va, epochs=15, lr=3e-3, verbose=False)
        print(f"{nombre}: exactitud validación final = {h['val_acc'][-1]:.3f}")

## 6. Keras 3 y JAX: el resto del ecosistema

**Keras 3** (2023→) es una API de alto nivel *multi-backend*: el mismo modelo corre sobre JAX, TensorFlow o PyTorch cambiando `KERAS_BACKEND`. Es la vía moderna para el código "TensorFlow" de Géron; TensorFlow 2 puro ha perdido tracción fuera de despliegues legacy y móviles (TFLite → LiteRT).

```python
import os; os.environ["KERAS_BACKEND"] = "jax"     # o "torch" / "tensorflow"
import keras
model = keras.Sequential([keras.layers.Input((64,)), keras.layers.Dense(128, activation="relu"),
                          keras.layers.Dropout(0.2), keras.layers.Dense(10)])
model.compile(optimizer=keras.optimizers.AdamW(3e-3), loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=["accuracy"])
model.fit(Xa, ya, validation_data=(Xb_, yb_), epochs=20, batch_size=64)
```

**JAX** (+ Flax NNX / Equinox / Optax): transformaciones funcionales `grad`, `jit`, `vmap`, `pmap`; dominante en investigación de gran escala (DeepMind, Anthropic, xAI) y en TPUs. Vale la pena aprender sus primitivas aunque tu día a día sea PyTorch.

In [ ]:
# OPCIONAL: Keras 3 sobre el backend que esté disponible
try:
    import os; os.environ.setdefault("KERAS_BACKEND", "torch" if TORCH else "tensorflow")
    import keras
    print("Keras", keras.__version__, "backend:", keras.backend.backend())
except Exception as e:
    print("Keras 3 no disponible (`uv pip install keras`):", type(e).__name__)

## 📦 Recursos de los repositorios oficiales de los libros

| Libro | Recurso | Para qué usarlo aquí |
|---|---|---|
| Weidman, *Deep Learning from Scratch* | [`01_foundations/`](https://github.com/SethHWeidman/DLFS_code/tree/master/01_foundations) (Code + Math), [`02_fundamentals/`](https://github.com/SethHWeidman/DLFS_code/tree/master/02_fundamentals), [`03_dlfs/`](https://github.com/SethHWeidman/DLFS_code/tree/master/03_dlfs), [`04_extensions/`](https://github.com/SethHWeidman/DLFS_code/tree/master/04_extensions) (optimizadores, dropout, inicialización), [`05_convolutions/`](https://github.com/SethHWeidman/DLFS_code/tree/master/05_convolutions) (incluye `Numpy_Convolution_Demos.ipynb`: convolución multicanal en NumPy puro), [`06_rnns/`](https://github.com/SethHWeidman/DLFS_code/tree/master/06_rnns), [`07_PyTorch/`](https://github.com/SethHWeidman/DLFS_code/tree/master/07_PyTorch) · librería [`lincoln/`](https://github.com/SethHWeidman/DLFS_code/tree/master/lincoln) (capas, operaciones, optimizadores como clases reutilizables) | Cada capítulo trae un notebook *Code* y otro *Math*: la versión "de libro" de nuestra sección 1. `lincoln` muestra cómo se estructura un mini-framework (lo que PyTorch hace por dentro) |
| Géron, *Hands-On ML 3* | [`10_neural_nets_with_keras.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/10_neural_nets_with_keras.ipynb), [`11_training_deep_neural_networks.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/11_training_deep_neural_networks.ipynb) (inicialización, BN, *schedulers*, regularización), [`14_deep_computer_vision_with_cnns.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/14_deep_computer_vision_with_cnns.ipynb), [`15_processing_sequences_using_rnns_and_cnns.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/15_processing_sequences_using_rnns_and_cnns.ipynb), [`extra_gradient_descent_comparison.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/extra_gradient_descent_comparison.ipynb), [`extra_ann_architectures.ipynb`](https://colab.research.google.com/github/ageron/handson-ml3/blob/main/extra_ann_architectures.ipynb) | Misma caja de herramientas de la sección 3, en Keras. Ejecútalos con `KERAS_BACKEND=torch` para ver que Keras 3 corre sobre PyTorch |
| Zhang et al., *D2L* | [Cap. 5 MLP](https://d2l.ai/chapter_multilayer-perceptrons/index.html), [Cap. 6 Builders' Guide](https://d2l.ai/chapter_builders-guide/index.html), [Cap. 7-8 CNN](https://d2l.ai/chapter_convolutional-neural-networks/index.html), [Cap. 9-10 RNN](https://d2l.ai/chapter_recurrent-neural-networks/index.html), [Cap. 12 Optimization](https://d2l.ai/chapter_optimization/index.html) | Cada sección tiene pestañas PyTorch/JAX/TF y ejercicios; el Cap. 12 deriva Adam, RMSProp y los *schedulers* con rigor |

## 7. Ejercicios
1. **Softmax desde cero.** Extiende la clase `MLP` de NumPy a $K$ clases con softmax + entropía cruzada categórica (la derivada respecto a los logits vuelve a ser $\hat p - \mathbf{y}$). Entrénala sobre `digits`.
2. **Ablación.** En la CNN, quita BatchNorm, luego Dropout, luego ambos. Repite 3 semillas cada uno y reporta media ± desviación de la exactitud. ¿Qué componente importa más aquí?
3. **Schedulers.** Sustituye `OneCycleLR` por `CosineAnnealingLR` con *warmup* lineal de 2 épocas y por una tasa constante. Grafica las tres curvas de validación.
4. **Residuales.** Construye un MLP de 20 capas con y sin conexiones residuales (`x + f(x)`) y compara si entrenan.
5. **Secuencias largas.** Aumenta `T` a 100 y 200 en la tarea de conteo. ¿A qué longitud deja de aprender la GRU? Guarda el resultado: lo compararás con un Transformer en el Módulo 5.
6. **Precisión mixta (si tienes GPU).** Mide el tiempo por época de la CNN con y sin `torch.autocast(dtype=torch.bfloat16)`.

## Referencias
* Weidman, S. (2019). *Deep Learning from Scratch*. O'Reilly. Cap. 1-7. Código: github.com/SethHWeidman/DLFS_code
* Zhang, A. et al. *Dive into Deep Learning*. Cap. 3-12. https://d2l.ai
* Géron, A. (2022). *Hands-On ML* (3.ª ed.). Cap. 10-11, 14-15.
* He, K. et al. (2016). *Deep Residual Learning for Image Recognition*. CVPR.
* Loshchilov, I. & Hutter, F. (2019). *Decoupled Weight Decay Regularization* (AdamW). ICLR.
* Liu, Z. et al. (2022). *A ConvNet for the 2020s* (ConvNeXt). CVPR.
* PyTorch 2.x docs: https://docs.pytorch.org · Keras 3: https://keras.io/keras_3/